In [1]:
import torch 
import torchvision 
import torch.nn as nn 
import torch.nn.functional as F  
from torch.utils.data import Subset
import torchvision.transforms.v2 as v2
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms.functional as TF

import numpy as np 
import pandas as pd 
from tqdm import tqdm 
import matplotlib as mpl 
import matplotlib.pyplot as plt 
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

import os
import glob 
import logging
import datetime
from os import cpu_count
from string import hexdigits
from multiprocessing import Manager
from multiprocessing.dummy import Pool

IMAGE_SIZE = 320

df = pd.read_csv('../malware-dataset.csv')

In [2]:
def make_image(file_path):
    binary = open(file_path, "rb").read()

    if not binary: 
        raise Exception("Zero data read")

    num_bytes = len(binary)
    shape = np.ceil(np.sqrt(num_bytes)).astype(np.int32)

    image = np.zeros((shape**2,))
    image[:num_bytes] = list(binary)
    image = torch.tensor(image).reshape((1, 1, shape, shape))
    return TF.resize(image, size=[IMAGE_SIZE, IMAGE_SIZE])

In [3]:
def init_shared_dict(shared_dict_):
    global shared_dict
    shared_dict = shared_dict_ 

def process_item(args):
    path, label = args
    try: 
        image = make_image(path).to(torch.uint8) / 255.0
    except: 
        return 
    filename = "".join(np.random.choice(list(hexdigits), 16))
    
    if label == 1: 
        save_path = os.path.join('./data/malicious/', f'{filename}.png')
    else: 
        save_path = os.path.join('./data/benign/', f'{filename}.png')

    shared_dict[path] = save_path

    torchvision.utils.save_image(image, save_path)

In [4]:
if len(glob.glob("../data/**/*.png", recursive=True)) == 0: 
    paths = df['0']
    labels = df['1']

    paths_labels = [tuple(x) for x in np.array([paths, labels]).T.tolist()]
    with Manager() as m: 
        shared_dict = m.dict() 
        with Pool(processes=cpu_count(), initializer=init_shared_dict, initargs=(shared_dict,)) as pool:
            list(tqdm(pool.imap_unordered(process_item, paths_labels), total=len(paths_labels)))
        lookup = dict(shared_dict)
        lookup = pd.DataFrame({"original": list(lookup.keys()), "image": list(lookup.values())})
        lookup.to_csv("lookup.csv")

In [5]:
if os.path.exists('../models/trained-cnn-malware-detector.pt'):
    model = torch.load("../models/trained-cnn-malware-detector.pt", weights_only=False)
    model = model.to('mps')
else: 
    model = torchvision.models.efficientnet_b0()
    model.set_submodule('classifier.1', nn.Linear(in_features=1280, out_features=2))
    model = model.to('mps')

In [7]:
transform = v2.Compose([
    v2.ToImage(), 
    v2.ToDtype(torch.float32)
])

def is_selected(path):
    return True 
    # return path in df['image'].tolist()

dataset = ImageFolder('../data', transform=transform, is_valid_file=is_selected)
idxs = list(range(len(dataset)))

train_idxs, test_idxs = train_test_split(idxs, stratify=dataset.targets)
train = Subset(dataset, train_idxs)
test = Subset(dataset, test_idxs)

train_loader = DataLoader(train, batch_size=32)
test_loader = DataLoader(test, batch_size=32)

In [7]:
EPOCHS = 20 
criterion = nn.BCEWithLogitsLoss()
optim = torch.optim.AdamW(model.parameters())

In [ ]:
def train(loader, model, optim, criterion, epoch=0):
    train_loss = []
    train_loop = tqdm(loader, leave=False)

    train_loop.set_description_str(f"Train {epoch + 1:02}/{EPOCHS}")

    for img, label in train_loop:
        img = img.to('mps')
        label = F.one_hot(label, 2).to(torch.float32).to('mps')
        
        optim.zero_grad() 
        output = model(img)
        break 
    
        loss = criterion(output, label)
        train_loss.append(loss)
        loss.backward() 
        optim.step()

    return sum(train_loss) / len(train_loss)

def test(loader, model, criterion, epoch=0):
    model.eval()

    loop = tqdm(loader, leave=False)
    predicted = []
    actual = []
    test_loss = []

    loop.set_description_str(f"Test epoch {epoch+1:02}/{EPOCHS}")

    for img, label in loop: 
        img = img.to('mps')    
        label = F.one_hot(label, 2).to(torch.float32).to('mps')

        with torch.no_grad():
            output = model(img)

        loss = criterion(output, label)
        test_loss.append(loss)

        y_hat = output.argmax(dim=-1)
        y = label.argmax(dim=-1)

        predicted.append(y_hat)
        actual.append(y)

    predicted = torch.hstack(predicted).cpu()
    actual = torch.hstack(actual).cpu()

    model.train()
    return sum(test_loss) / len(test_loss), classification_report(actual, predicted, output_dict=True)
        

train_loss = []
test_loss = []

if not os.path.exists("../models/trained-cnn-malware-detector.pt"):
    for i in range(EPOCHS):
        loss = train(train_loader, model, optim, criterion, i)
        train_loss.append(loss)

        loss, report = test(test_loader, model, criterion, i)
        test_loss.append(loss)

KeyboardInterrupt: 

In [ ]:
plt.figure(dpi=300)
plt.plot(torch.tensor(train_loss), label="Train")
plt.plot(torch.tensor(test_loss), label="Test")
plt.xlabel("Episode")
plt.ylabel("Average BCE Loss")

In [ ]:
predicted_logits = []
actual = []

for img, label in tqdm(test_loader): 
    img = img.to('mps')    
    label = F.one_hot(label, 2).to(torch.float32).to('mps')
    
    with torch.no_grad():
        output = model(img)

    loss = criterion(output, label)

    y = label.argmax(dim=-1)

    predicted_logits.append(output)
    actual.append(y)

predicted_logits = torch.vstack(predicted_logits).cpu()
actual = torch.hstack(actual).cpu()

  0%|          | 0/87 [00:00<?, ?it/s]

torch.Size([32, 3, 320, 320])


  0%|          | 0/87 [00:04<?, ?it/s]


RuntimeError: vstack expects a non-empty TensorList

In [ ]:
from sklearn.metrics import auc, roc_curve

pos_logits = predicted_logits[:, 1]
probs = F.sigmoid(pos_logits)

fpr, tpr, _ = roc_curve(actual, probs)
roc_auc = auc(fpr, tpr)
plt.figure(dpi=200)
plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], linestyle='--', c='grey')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.grid(True)
plt.annotate(f"AUC = {roc_auc:.2f}", xy=(0.85, 0), bbox=dict(facecolor='#eee'))
plt.show()

In [ ]:
y_hat = predicted_logits.argmax(dim=-1)
y = actual

print(classification_report(y, y_hat))

In [ ]:
m = []
b = []

for imgs, labels in tqdm(test_loader):
    imgs = imgs.cpu()
    labels = labels.cpu()

    mal = imgs[labels == 1]
    ben = imgs[labels == 0]
    
    m.extend(mal)
    b.extend(ben)
m = torch.stack(m)
b = torch.stack(b)

In [ ]:
mu_m = m.mean(dim=0).permute(1, 2, 0)
std_m = m.std(dim=0).permute(1, 2, 0)
mu_b = b.mean(dim=0).permute(1, 2, 0)
std_b = b.std(dim=0).permute(1, 2, 0)
mu_mse = F.mse_loss(mu_m, mu_b, reduction="none")
std_mse = F.mse_loss(std_m, std_b, reduction="none")

In [ ]:
fig, axs = plt.subplots(2, 3, dpi=300)
axs[0, 0].title.set_text("Malicious")
axs[0, 0].imshow((mu_m / 255.0).mean(dim=-1), cmap="viridis")
axs[0, 0].set_xticks([])
axs[0, 0].set_yticks([])
axs[0, 1].title.set_text("Benign")
axs[0, 1].imshow((mu_b / 255.0).mean(dim=-1), cmap="viridis")
axs[0, 1].set_xticks([])
axs[0, 1].set_yticks([])
axs[0, 2].title.set_text("MSE")
axs[0, 2].imshow(((mu_mse - mu_mse.min()) / (mu_mse.max() - mu_mse.min())).mean(dim=-1), cmap="viridis")
axs[0, 2].set_xticks([])
axs[0, 2].set_yticks([])
axs[1, 0].imshow((std_m / 255.0).mean(dim=-1), cmap="viridis")
axs[1, 0].set_xticks([])
axs[1, 0].set_yticks([])
axs[1, 1].imshow((std_b / 255.0).mean(dim=-1), cmap="viridis")
axs[1, 1].set_xticks([])
axs[1, 1].set_yticks([])
axs[1, 2].imshow(((std_mse - std_mse.min()) / (std_mse.max() - std_mse.min())).mean(dim=-1), cmap="viridis")
axs[1, 2].set_xticks([])
axs[1, 2].set_yticks([])
plt.show()

In [ ]:
import cv2

incorrect = lookup.iloc[torch.argwhere(y_hat != y).flatten()]
img = np.zeros((len(incorrect), 320, 320, 3))
for i, path in enumerate(incorrect.image):
    img[i] = cv2.imread(path) 

In [ ]:
plt.figure(dpi=300)
plt.title("Average incorrect representation")
plt.axis('off')
plt.subplot(1,2,1)
plt.title("Mean")
plt.imshow(img.mean(axis=(0, -1), keepdims=True)[0])
plt.axis('off')
plt.subplot(1,2,2)
plt.title("Variance")
plt.imshow(img.std(axis=(0, -1), keepdims=True)[0])
plt.axis('off')
plt.show()